# JS06: Tugas Lab

## Soal 1: SVM pada voice.csv

Gunakan kernel linier, polynomial, dan RBF pada split 70:30 serta 80:20. Tabulasikan akurasi setiap kombinasi.

## Persiapan data suara

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen

voice_path = Path('voice.csv')
if not voice_path.exists():
    request = Request('https://1812311909-files.gitbook.io/~/files/v0/b/gitbook-x-prod.appspot.com/o/spaces%2FYHOv2XH8FJMJSMsnhwNa%2Fuploads%2FVjixBjsPUXz1tYK1bRXm%2Fvoice.csv?alt=media&token=c9d523b3-4b2f-4188-aaa5-0ca06e647da1', headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=120) as response:
        voice_path.write_bytes(response.read())
print(f'Dataset siap: {voice_path.name}')

Dataset siap: voice.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from IPython.display import display

voice = pd.read_csv(voice_path)
print('Ukuran data:', voice.shape)
print(voice['label'].value_counts())
voice.head()

Ukuran data: (3168, 21)
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


## Perbandingan split dan kernel

Semua 20 fitur numerik digunakan. Standardisasi hanya dipelajari dari data latih. Kedua split memakai random state 42 dan stratifikasi label. Kernel polynomial menggunakan degree 3, sedangkan parameter model lainnya mengikuti nilai bawaan.

In [3]:
X_voice = voice.drop(columns='label')
y_voice = voice['label']
voice_results = []
voice_models = {}

for ratio, test_size in [('70:30', 0.3), ('80:20', 0.2)]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_voice, y_voice, test_size=test_size, random_state=42, stratify=y_voice
    )
    for kernel in ['linear', 'poly', 'rbf']:
        model = make_pipeline(StandardScaler(), SVC(kernel=kernel))
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        voice_models[(ratio, kernel)] = model
        voice_results.append({
            'Split': ratio,
            'Kernel': kernel,
            'Data latih': len(y_train),
            'Data uji': len(y_test),
            'Akurasi latih': accuracy_score(y_train, train_pred),
            'Akurasi uji': accuracy_score(y_test, test_pred),
        })

voice_results = pd.DataFrame(voice_results)
print(voice_results.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
best_voice = voice_results.loc[voice_results['Akurasi uji'].idxmax()]
print(f"Akurasi uji tertinggi: {best_voice['Kernel']}, split {best_voice['Split']}, "
      f"akurasi {best_voice['Akurasi uji']:.6f}")

Split Kernel  Data latih  Data uji  Akurasi latih  Akurasi uji
70:30 linear        2217       951       0.972034     0.978970
70:30   poly        2217       951       0.966622     0.960042
70:30    rbf        2217       951       0.983762     0.983176
80:20 linear        2534       634       0.976717     0.974763
80:20   poly        2534       634       0.968035     0.955836
80:20    rbf        2534       634       0.983820     0.982650
Akurasi uji tertinggi: rbf, split 70:30, akurasi 0.983176


## Soal 2: SVM RBF dengan fitur histogram

Gunakan gambar dari Lab 5, fitur histogram, dan split 80:20. Catat akurasinya. Percobaan ini juga membandingkan beberapa nilai C dan gamma pada data latih.

## Persiapan data gambar

In [4]:
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile
import shutil

image_root = Path('images')
if not all((image_root / split / label).is_dir()
           for split in ['training', 'test'] for label in ['day', 'night']):
    archive = Path('images.zip')
    if not archive.exists():
        request = Request('https://1812311909-files.gitbook.io/~/files/v0/b/gitbook-x-prod.appspot.com/o/spaces%2FYHOv2XH8FJMJSMsnhwNa%2Fuploads%2F9IvI7imHRxFYT9gtRGh7%2Fimages.zip?alt=media&token=d1bc0a22-4056-4a11-a801-04f245ee0e62', headers={'User-Agent': 'Mozilla/5.0'})
        with urlopen(request, timeout=120) as response, archive.open('wb') as target:
            shutil.copyfileobj(response, target)
    with ZipFile(archive) as zipped:
        members = [name for name in zipped.namelist() if name.startswith('images/')]
        zipped.extractall('.', members=members)

for split in ['training', 'test']:
    for label in ['day', 'night']:
        count = len(list((image_root / split / label).glob('*.jpg')))
        print(f'{split} {label}: {count} gambar')

training day: 120 gambar
training night: 120 gambar
test day: 80 gambar
test night: 80 gambar


## Ekstraksi histogram

Semua gambar pada folder training dan test digabung, kemudian dibagi ulang 80:20 sesuai soal. Setiap gambar berukuran 1100 × 600 piksel. Fitur terdiri dari histogram H, S, dan V, masing masing 32 bin yang dinormalisasi, sehingga setiap gambar memiliki 96 fitur.

In [5]:
import cv2
import matplotlib.image as mpimg

def histogram_feature(image):
    image = cv2.resize(image, (1100, 600))
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    features = []
    for channel, upper_bound in [(0, 180), (1, 256), (2, 256)]:
        hist = cv2.calcHist([hsv], [channel], None, [32], [0, upper_bound]).ravel()
        hist = hist / hist.sum()
        features.extend(hist)
    return np.asarray(features, dtype=np.float32)

image_features = []
image_labels = []
image_paths = []
for split in ['training', 'test']:
    for label in ['day', 'night']:
        for path in sorted((Path('images') / split / label).glob('*.jpg')):
            image_features.append(histogram_feature(mpimg.imread(path)))
            image_labels.append(1 if label == 'day' else 0)
            image_paths.append(str(path))

X_images = np.asarray(image_features)
y_images = np.asarray(image_labels)
print('Ukuran fitur histogram:', X_images.shape)
print('Jumlah day:', np.sum(y_images == 1))
print('Jumlah night:', np.sum(y_images == 0))
X_train, X_test, y_train, y_test = train_test_split(
    X_images, y_images, test_size=0.2, random_state=42, stratify=y_images
)
print('Jumlah data latih dan uji:', len(y_train), len(y_test))

Ukuran fitur histogram: (400, 96)
Jumlah day: 200
Jumlah night: 200
Jumlah data latih dan uji: 320 80


## Pelatihan model dan tuning

GridSearchCV memakai lima fold berstratifikasi pada data latih. Data uji hanya digunakan untuk evaluasi akhir.

In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

histogram_model = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
histogram_params = {
    'svc__C': [1, 10, 100],
    'svc__gamma': ['scale', 0.01, 0.1],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
histogram_search = GridSearchCV(histogram_model, histogram_params, scoring='accuracy', cv=cv)
histogram_search.fit(X_train, y_train)
print('Parameter terbaik:', histogram_search.best_params_)
print('Akurasi validasi:', histogram_search.best_score_)

Parameter terbaik: {'svc__C': 10, 'svc__gamma': 'scale'}
Akurasi validasi: 0.996875


## Evaluasi histogram

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

histogram_best = histogram_search.best_estimator_
histogram_train_pred = histogram_best.predict(X_train)
histogram_test_pred = histogram_best.predict(X_test)
print('Akurasi latih histogram:', accuracy_score(y_train, histogram_train_pred))
print('Akurasi uji histogram:', accuracy_score(y_test, histogram_test_pred))
print('Confusion matrix histogram (night, day):')
print(confusion_matrix(y_test, histogram_test_pred))
print(classification_report(y_test, histogram_test_pred, target_names=['night', 'day']))

Akurasi latih histogram: 1.0
Akurasi uji histogram: 0.975
Confusion matrix histogram (night, day):
[[40  0]
 [ 2 38]]
              precision    recall  f1-score   support

       night       0.95      1.00      0.98        40
         day       1.00      0.95      0.97        40

    accuracy                           0.97        80
   macro avg       0.98      0.97      0.97        80
weighted avg       0.98      0.97      0.97        80

